# Neural Radiance Fields (NeRF)

Wiki reference for [neural radiance fields](https://ml-viz-ruby.vercel.app/wiki/neural-radiance-fields).

**The idea in one sentence.** A NeRF is an MLP that maps a 3D point to color and density, rendered
into an image by **volume rendering** (compositing samples along each ray) — and it only works
because of **positional encoding**: mapping coordinates through sinusoids at many frequencies,
which defeats the MLP's **spectral bias** toward smooth, low-frequency functions.

We implement positional encoding and volume rendering from scratch, **validate the encoding
dimensionality and the render weights**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': True,
})
np.random.seed(0)

## 1. Positional encoding — the key trick

$$\gamma(p) = (\sin(2^0\pi p), \cos(2^0\pi p), \ldots, \sin(2^{L-1}\pi p), \cos(2^{L-1}\pi p))$$

We fit a small MLP to a high-frequency target signal, with and without positional encoding.

In [ ]:
def positional_encoding(x, L=6):
    """Map scalar input x ∈ [0,1] to [sin/cos at L frequencies]."""
    x = x[:, None]  # (n, 1)
    feats = [x]
    for i in range(L):
        freq = 2.0 ** i * np.pi
        feats.append(np.sin(freq * x))
        feats.append(np.cos(freq * x))
    return np.concatenate(feats, axis=1)

# Target: high-frequency signal
def target_fn(x):
    return 0.5 + 0.3*np.sin(20*x) + 0.2*np.sin(50*x)

x_train = np.linspace(0, 1, 200)
y_train = target_fn(x_train)

# Tiny 2-layer MLP trained with numpy gradient descent
class MLP:
    def __init__(self, in_dim, hidden=64):
        s = np.sqrt(2.0 / in_dim)
        self.W1 = np.random.randn(in_dim, hidden) * s
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, hidden) * np.sqrt(2.0/hidden)
        self.b2 = np.zeros(hidden)
        self.W3 = np.random.randn(hidden, 1) * np.sqrt(2.0/hidden)
        self.b3 = np.zeros(1)

    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1; self.a1 = np.maximum(0, self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2; self.a2 = np.maximum(0, self.z2)
        self.out = self.a2 @ self.W3 + self.b3
        return self.out[:, 0]

    def train_step(self, X, y, lr=0.01):
        n = len(y)
        pred = self.forward(X)
        dout = (2.0/n) * (pred - y)[:, None]
        dW3 = self.a2.T @ dout; db3 = dout.sum(0)
        da2 = dout @ self.W3.T; dz2 = da2 * (self.z2 > 0)
        dW2 = self.a1.T @ dz2; db2 = dz2.sum(0)
        da1 = dz2 @ self.W2.T; dz1 = da1 * (self.z1 > 0)
        dW1 = self.X.T @ dz1; db1 = dz1.sum(0)
        for p, g in [(self.W1,dW1),(self.b1,db1),(self.W2,dW2),(self.b2,db2),(self.W3,dW3),(self.b3,db3)]:
            p -= lr * g
        return np.mean((pred - y)**2)

# Train WITHOUT positional encoding (raw x)
X_raw = x_train[:, None]
mlp_raw = MLP(in_dim=1)
for _ in range(3000): mlp_raw.train_step(X_raw, y_train, lr=0.05)

# Train WITH positional encoding
X_pe = positional_encoding(x_train, L=6)
mlp_pe = MLP(in_dim=X_pe.shape[1])
for _ in range(3000): mlp_pe.train_step(X_pe, y_train, lr=0.05)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_train, y_train, color='white', linewidth=2, alpha=0.6, label='Target (high-freq)')
ax.plot(x_train, mlp_raw.forward(X_raw), color='#f43f5e', linewidth=2, label='MLP, raw input (blurry)')
ax.plot(x_train, mlp_pe.forward(X_pe), color='#10b981', linewidth=2, label='MLP + positional encoding (sharp)')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Positional Encoding Lets an MLP Fit High-Frequency Detail', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final MSE — raw input:        {np.mean((mlp_raw.forward(X_raw)-y_train)**2):.5f}")
print(f"Final MSE — pos. encoding:    {np.mean((mlp_pe.forward(X_pe)-y_train)**2):.5f}")

### Validate: positional encoding expands the input

Positional encoding maps a scalar to $1 + 2L$ features (the raw value plus $\sin/\cos$ at $L$
frequencies), giving the MLP the high-frequency basis it needs. We confirm the dimensionality.

In [ ]:
pe = positional_encoding(x_train, L=6)
print(f'positional encoding maps a scalar -> {pe.shape[1]} features (expected 1 + 2*6 = 13)')
assert pe.shape[1] == 1 + 2 * 6, 'positional encoding = raw value + sin/cos at L frequencies'
print('\n✅ positional encoding gives the MLP a high-frequency Fourier basis')

## 2. Volume rendering along a ray

$$C = \sum_i T_i (1 - e^{-\sigma_i \delta_i}) c_i, \quad T_i = \exp\left(-\sum_{j<i}\sigma_j \delta_j\right)$$

We simulate a ray passing through a scene with a solid opaque surface and accumulate color.

In [ ]:
def volume_render(densities, colors, deltas):
    """
    densities: (N,) sigma per sample along ray
    colors:    (N,) scalar color per sample (grayscale for simplicity)
    deltas:    (N,) distance between samples
    Returns accumulated color and the per-sample weights.
    """
    alpha = 1 - np.exp(-densities * deltas)             # opacity per sample
    T = np.concatenate([[1.0], np.cumprod(1 - alpha + 1e-10)[:-1]])  # transmittance
    weights = T * alpha
    color = np.sum(weights * colors)
    return color, weights

N = 100
t = np.linspace(0, 1, N)
deltas = np.full(N, 1.0/N)

# Scene: empty until t=0.5, then an opaque red surface
densities = np.where(t > 0.5, 30.0, 0.0)
densities += np.where((t > 0.7) & (t < 0.75), 50.0, 0.0)  # a second thin surface (occluded)
colors = np.where(t > 0.5, 0.9, 0.0)
colors = np.where((t > 0.7) & (t < 0.75), 0.2, colors)

color, weights = volume_render(densities, colors, deltas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t, densities, color='#f59e0b', linewidth=2)
axes[0].set_xlabel('Distance along ray t'); axes[0].set_ylabel('Density σ')
axes[0].set_title('Scene Density Along Ray', color='#e2e8f0')
axes[1].plot(t, weights, color='#22d3ee', linewidth=2)
axes[1].fill_between(t, weights, color='#22d3ee', alpha=0.2)
axes[1].set_xlabel('Distance along ray t'); axes[1].set_ylabel('Rendering weight T·α')
axes[1].set_title('Weights Peak at First Surface (occlusion handled)', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"Rendered pixel color: {color:.3f}")
print("Notice the second surface (t≈0.72) gets ~0 weight — it is occluded by the first.")

### Validate: volume rendering composites along the ray

Volume rendering turns per-sample densities and colors into one pixel: opacity $\alpha = 1 -
e^{-\sigma\delta}$, transmittance $T$ accumulates, and the weights $w = T\alpha$ form a valid
(sub-)distribution — an opaque sample **occludes** everything behind it. We confirm the weights
are valid and peak at the first opaque surface.

In [ ]:
N = 8
dens = np.zeros(N); dens[2] = 100.0   # one opaque surface at sample 2
cols = np.linspace(0, 1, N); deltas = np.ones(N) * 0.1
_, w = volume_render(dens, cols, deltas)
print(f'weights sum = {w.sum():.3f}, argmax = {w.argmax()} (the opaque surface)')
assert (w >= 0).all() and w.sum() <= 1 + 1e-9, 'render weights form a valid sub-distribution along the ray'
assert w.argmax() == 2, 'the first opaque surface gets the most weight; samples behind it are occluded'
print('\n✅ volume rendering composites and handles occlusion via transmittance')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no positional encoding** | spectral bias -> blurry results (demo) |
| **too many frequencies** | can introduce noise / aliasing |
| **slow rendering** | many MLP evaluations per ray per pixel — costly |
| **per-scene training** | a NeRF fits ONE scene; it doesn't generalize |
| **sampling along rays** | coarse sampling misses thin surfaces — hierarchical sampling helps |

Demo: the raw-input MLP fails on a high-frequency target that positional encoding fits.

In [ ]:
# Why positional encoding is ESSENTIAL: an MLP on raw coordinates has SPECTRAL BIAS — it fits
# smooth, low-frequency functions easily but cannot represent high-frequency detail. On a
# high-frequency target the raw-input MLP fails while the positionally-encoded one succeeds. We
# compare their final errors.
mse_raw = np.mean((mlp_raw.forward(X_raw) - y_train) ** 2)
mse_pe = np.mean((mlp_pe.forward(X_pe) - y_train) ** 2)
print(f'final MSE: raw input = {mse_raw:.5f}   positional encoding = {mse_pe:.5f}')
assert mse_raw > 10 * mse_pe, 'without positional encoding the MLP suffers spectral bias and cannot fit high frequencies'
print('\nRaw-coordinate MLPs are biased toward smoothness -> positional encoding is what makes NeRF sharp.')

## ✏️ Your turn

**Exercise 1 — Encoding frequency bands.** Re-fit the positional-encoding MLP with L ∈ {1, 2, 4, 8} frequency bands. Plot final MSE vs L. Too few bands underfit; do very many bands help or introduce noise?

In [ ]:
L_values = [1, 2, 4, 8]
# TODO(you): for each L, build encoding, train an MLP, record final MSE

In [ ]:
# Assert cell
mses = []
for L in L_values:
    Xl = positional_encoding(x_train, L=L)
    m = MLP(in_dim=Xl.shape[1])
    for _ in range(2000): m.train_step(Xl, y_train, lr=0.05)
    mses.append(np.mean((m.forward(Xl) - y_train)**2))
    print(f"L={L}: final MSE = {mses[-1]:.5f}")
assert mses[-1] < mses[0], "More frequency bands should reduce error on a high-freq target"

<details><summary>Solution</summary>

```python
mses = []
for L in L_values:
    Xl = positional_encoding(x_train, L=L)
    m = MLP(in_dim=Xl.shape[1])
    for _ in range(2000): m.train_step(Xl, y_train, lr=0.05)
    mses.append(np.mean((m.forward(Xl) - y_train)**2))

plt.plot(L_values, mses, 'o-', color='#6366f1', linewidth=2)
plt.xlabel('Frequency bands L'); plt.ylabel('Final MSE')
plt.title('More Bands → Sharper Fit (up to a point)')
plt.show()
```

Error drops as L increases because the highest target frequency (sin 50x) needs enough bands to be representable. Beyond what the signal contains, extra bands give diminishing returns and can add high-frequency noise/aliasing — which is exactly what Mip-NeRF later addressed with scale-aware encodings.

</details>

## Key takeaways

- **NeRF = MLP(point) -> (color, density),** rendered by volume compositing along rays.
- **Positional encoding** expands coordinates into a Fourier basis (verified dimensionality).
- **Volume rendering** composites samples and handles occlusion via transmittance (verified).
- **Spectral bias:** without positional encoding the MLP can't fit high-frequency detail (demo).